In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [2]:
import sys, os
import pandas as pd

sys.path.append(os.environ['LP_ROOT'])
import project

sys.path.append(os.environ['PROJECT'] + '/param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [3]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
for p in data_root.iterdir():
    print(p)

/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/subject_files.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Images
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-17.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/DISK_USAGE
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/ClinicalData


In [4]:
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
subject_list

['16514P',
 '20748Q',
 '11007Z',
 '14771Z',
 '13651K',
 '15900P',
 '15312Y',
 '21042H',
 '21877G',
 '10127E',
 '23577E',
 '13816Q',
 '21559S',
 '25335Q',
 '23023N',
 '10572Z',
 '10887Y',
 '21410K',
 '17920F',
 '14684E',
 '16132B',
 '17862R',
 '11746L',
 '25728J',
 '20609C',
 '16637F',
 '13297S',
 '25767T',
 '25695U',
 '21572K',
 '15078Q',
 '15297C',
 '14857J',
 '13460D',
 '18840M',
 '12506W',
 '11498S',
 '21611U',
 '15088T',
 '12831H',
 '19612E',
 '19784H',
 '10217F',
 '23037Y',
 '25130Y',
 '15623P',
 '19027T',
 '14380K',
 '10212V',
 '16060C',
 '19356M',
 '22357L',
 '25923H',
 '21189L',
 '24608U',
 '15204V',
 '11743F',
 '11879E',
 '12477P',
 '10815Z',
 '17790S',
 '15489L',
 '11850G',
 '13034M',
 '17255W',
 '14632L',
 '26066U',
 '18184E',
 '15532M',
 '19558Y',
 '20640W',
 '21741H',
 '18015H',
 '17137Q',
 '12422Q',
 '15699W',
 '21933Q',
 '18397V',
 '14550J',
 '20703U',
 '21004Z',
 '24581A',
 '24331D',
 '17505T',
 '15894U',
 '16977D',
 '18935X',
 '19410S',
 '18515B',
 '21399W',
 '21899Q',

In [9]:
base_dir = '2026-08-08_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=16G
#$ -l h_rt=12:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects=[{subject}] \\
    --set dataset.examples.variant={variant} \\

'''
name_format = 'job{params_hash}'

grid = ps.param_grid(
    config='2026-08-08_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=subject_list,
    variant='2026-08-08'
)
len(grid)

1000

In [10]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,jobb5f34bb3ee104fea,RUNNING,1,7144818,scc-307,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,16514P,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,None,None
1,job592ea5b621a54700,RUNNING,1,7144819,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 13.7626s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,20748Q,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,None,None
2,job4ce868ed0e3b2485,RUNNING,1,7144820,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,\nRunning sliver perturbation...\nLegend of th...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,11007Z,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,None,None
3,job2536956436071621,RUNNING,1,7144821,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 10.589s\n\nRunning sl...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,14771Z,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,None,None
4,jobe2fe1b602d1ab0c6,RUNNING,1,7144822,scc-215,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 11.3248s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,13651K,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,jobad976677102cebf9,RECOVERED,1,7145823,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,"File ""/projectnb/batmanlab/mragoza/lung-proj...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,20519B,2026-08-08,NaN,None,None,None,None,False,None,None
996,job60e111a5f43e1752,RECOVERED,1,7145824,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 20.4356s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,12294H,2026-08-08,NaN,None,None,None,None,False,None,None
997,jobcfd159863ee1c7f4,RECOVERED,1,7145825,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 18.7251s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,23123R,2026-08-08,NaN,None,None,None,None,False,None,None
998,jobb993cdf66950d0c5,RECOVERED,1,7145826,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,\nRunning sliver perturbation...\nLegend of th...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,16546C,2026-08-08,NaN,None,None,None,None,False,None,None


In [11]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,jobb5f34bb3ee104fea,RUNNING,1,7144818,scc-307,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,16514P,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,<NA>,<NA>
1,job592ea5b621a54700,RUNNING,1,7144819,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 13.7626s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,20748Q,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,<NA>,<NA>
2,job4ce868ed0e3b2485,RUNNING,1,7144820,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,\nRunning sliver perturbation...\nLegend of th...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,11007Z,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,<NA>,<NA>
3,job2536956436071621,RUNNING,1,7144821,scc-j07,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 10.589s\n\nRunning sl...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,14771Z,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,<NA>,<NA>
4,jobe2fe1b602d1ab0c6,RUNNING,1,7144822,scc-215,00:00:16,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 11.3248s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,13651K,2026-08-08,NaN,2026-08-11T22:30:47,status,None,None,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,jobad976677102cebf9,RECOVERED,1,7145823,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,"File ""/projectnb/batmanlab/mragoza/lung-proj...",/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,20519B,2026-08-08,NaN,None,None,None,None,False,<NA>,<NA>
996,job60e111a5f43e1752,RECOVERED,1,7145824,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 20.4356s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,12294H,2026-08-08,NaN,None,None,None,None,False,<NA>,<NA>
997,jobcfd159863ee1c7f4,RECOVERED,1,7145825,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,Total optimization time: 18.7251s\n\nRunning s...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,23123R,2026-08-08,NaN,None,None,None,None,False,<NA>,<NA>
998,jobb993cdf66950d0c5,RECOVERED,1,7145826,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,\nRunning sliver perturbation...\nLegend of th...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,16546C,2026-08-08,NaN,None,None,None,None,False,<NA>,<NA>


In [12]:
jobs.groupby('job_state').count()

,job_name,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,script_path,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
job_state,,,,,,,,,,,,,,,,,,,,,
RECOVERED,976,976,976,0,0,976,976,976,976,976,...,976,976,0,0,0,0,0,976,0,0
RUNNING,24,24,24,24,24,24,24,24,24,24,...,24,24,0,24,24,0,0,24,0,0


In [77]:
# classify error messages
for j, row in jobs.iterrows():
    if pd.isnull(row.stderr):
        print(f'{j}: No stderr file')
    elif not str(row.stderr).strip():
        print(f'{j}: No error message')
    elif 'Non-positive density' in row.stderr:
        print(f'{j}: Non-positive density')
    elif 'CUDA out of memory' in row.stderr or ('Failed to allocate' in row.stderr and 'bytes on device \'cuda' in row.stderr):
        print(f'{j}: GPU out of memory')
    elif 'Exuding' in row.stderr or ('Moving #' in row.stderr and 'addr:' in row.stderr):
        print(f'{j}: pygalmesh output')
    else:
        print(f'{j}: Uncategorized')
        raise RuntimeError(f'Uncategorized stderr for job index {j}:\n{row.stderr}')

0: No error message
1: pygalmesh output
2: pygalmesh output
3: pygalmesh output
4: pygalmesh output
5: pygalmesh output
6: pygalmesh output
7: pygalmesh output
8: pygalmesh output
9: pygalmesh output
10: pygalmesh output
11: pygalmesh output
12: pygalmesh output
13: pygalmesh output
14: pygalmesh output
15: pygalmesh output
16: pygalmesh output
17: pygalmesh output
18: pygalmesh output
19: pygalmesh output
20: pygalmesh output
21: pygalmesh output
22: GPU out of memory
23: pygalmesh output
24: pygalmesh output
25: pygalmesh output
26: pygalmesh output
27: pygalmesh output
28: pygalmesh output
29: GPU out of memory
30: pygalmesh output
31: pygalmesh output
32: pygalmesh output
33: pygalmesh output
34: pygalmesh output
35: pygalmesh output
36: pygalmesh output
37: pygalmesh output
38: pygalmesh output
39: pygalmesh output
40: pygalmesh output
41: pygalmesh output
42: pygalmesh output
43: GPU out of memory
44: pygalmesh output
45: pygalmesh output
46: pygalmesh output
47: pygalmesh output

In [52]:
print(jobs[jobs.stderr.notna()].iloc[65].stderr)

  File "/projectnb/batmanlab/mragoza/lung-project/project/physics/warp/solver.py", line 412, in assemble_jacobian
    J = wp.fem.integrate(
        ^^^^^^^^^^^^^^^^^
  File "/projectnb/batmanlab/mragoza/mambaforge/envs/warp/lib/python3.11/site-packages/warp/_src/fem/integrate.py", line 2055, in integrate
    return _launch_integrate_kernel(
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/projectnb/batmanlab/mragoza/mambaforge/envs/warp/lib/python3.11/site-packages/warp/_src/fem/integrate.py", line 1751, in _launch_integrate_kernel
    local_result = cache.borrow_temporary(
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/projectnb/batmanlab/mragoza/mambaforge/envs/warp/lib/python3.11/site-packages/warp/_src/fem/cache.py", line 674, in borrow_temporary
    Temporary(shape=shape, dtype=dtype, pinned=pinned, device=device, requires_grad=requires_grad)
  File "/projectnb/batmanlab/mragoza/mambaforge/envs/warp/lib/python3.11/site-packages/warp/_src/types.py", line 3285, in __init__
    sel

In [25]:
jobs.loc[:, 'job_id'] = pd.NA

In [8]:
%autoreload
jobs = ps.submit(jobs)
jobs

NameError: name 'requires_id' is not defined